# Vision Transformer (ViT) for Image Classification

<!--
Author: RSK World
Website: https://rskworld.in
Email: help@rskworld.in
Phone: +91 93305 39277
-->

This notebook demonstrates the Vision Transformer (ViT) architecture for image classification. ViT splits images into patches and processes them as sequences using transformer architecture.

## Features
- Vision Transformer (ViT) architecture
- Patch-based image embedding
- Multi-head self-attention mechanism
- Positional encoding for spatial information
- State-of-the-art classification accuracy



In [ ]:
# Import necessary libraries
# Author: RSK World
# Website: https://rskworld.in

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import sys

# Add project root to path
sys.path.append('.')

from vit_model import VisionTransformer, create_vit_model
from utils import load_config, get_data_transforms

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")



## 1. Model Architecture

Let's create and examine the Vision Transformer model architecture.


In [ ]:
# Create ViT model
# Author: RSK World
# Website: https://rskworld.in

# Model configuration
config = {
    'image_size': 224,
    'patch_size': 16,
    'num_classes': 1000,
    'dim': 768,
    'depth': 12,
    'heads': 12,
    'mlp_dim': 3072,
    'dropout': 0.1,
    'emb_dropout': 0.1
}

# Create model
model = VisionTransformer(**config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model Architecture:")
print(f"  Image Size: {config['image_size']}x{config['image_size']}")
print(f"  Patch Size: {config['patch_size']}x{config['patch_size']}")
print(f"  Number of Patches: {(config['image_size'] // config['patch_size']) ** 2}")
print(f"  Embedding Dimension: {config['dim']}")
print(f"  Transformer Depth: {config['depth']}")
print(f"  Number of Heads: {config['heads']}")
print(f"  Number of Classes: {config['num_classes']}")
print(f"\nTotal Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")



## 2. Test Model Forward Pass

Let's test the model with a dummy input to verify it works correctly.


In [ ]:
# Test forward pass
# Author: RSK World
# Website: https://rskworld.in

# Create dummy input
dummy_input = torch.randn(1, 3, 224, 224)
print(f"Input shape: {dummy_input.shape}")

# Forward pass
model.eval()
with torch.no_grad():
    output = model(dummy_input)

print(f"Output shape: {output.shape}")
print(f"Output (first 10 values): {output[0, :10]}")

# Get predictions
probabilities = torch.nn.functional.softmax(output, dim=1)
top5_probs, top5_indices = torch.topk(probabilities, 5)

print(f"\nTop 5 Predictions:")
for i, (prob, idx) in enumerate(zip(top5_probs[0], top5_indices[0]), 1):
    print(f"  {i}. Class {idx.item()}: {prob.item() * 100:.2f}%")



## 3. Visualize Patch Embedding

Let's visualize how the image is split into patches.


In [ ]:
# Visualize patch embedding
# Author: RSK World
# Website: https://rskworld.in

def visualize_patches(image_path, patch_size=16, image_size=224):
    """
    Visualize how an image is divided into patches
    Author: RSK World
    Website: https://rskworld.in
    """
    # Load image
    img = Image.open(image_path).convert('RGB')
    img = img.resize((image_size, image_size))
    img_array = np.array(img)
    
    # Create figure
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Original image
    axes[0].imshow(img_array)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Image with patch grid
    axes[1].imshow(img_array)
    axes[1].set_title(f'Image with {patch_size}x{patch_size} Patches')
    axes[1].axis('off')
    
    # Draw patch grid
    num_patches = image_size // patch_size
    for i in range(num_patches + 1):
        # Vertical lines
        axes[1].axvline(i * patch_size, color='red', linewidth=0.5)
        # Horizontal lines
        axes[1].axhline(i * patch_size, color='red', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Image size: {image_size}x{image_size}")
    print(f"Patch size: {patch_size}x{patch_size}")
    print(f"Number of patches: {num_patches * num_patches}")

# Example usage (uncomment if you have an image)
# visualize_patches('path/to/your/image.jpg', patch_size=16, image_size=224)



## 4. Model Components

Let's examine the key components of the ViT architecture.


In [ ]:
# Examine model components
# Author: RSK World
# Website: https://rskworld.in

print("Model Components:")
print("=" * 50)

for name, module in model.named_children():
    print(f"\n{name}:")
    print(f"  Type: {type(module).__name__}")
    if hasattr(module, 'weight'):
        if module.weight is not None:
            print(f"  Weight shape: {module.weight.shape}")

# Check patch embedding
print("\n" + "=" * 50)
print("Patch Embedding Details:")
patch_embed = model.patch_embedding
print(f"  Input channels: 3")
print(f"  Embedding dimension: {patch_embed.projection.out_channels}")
print(f"  Patch size: {patch_embed.patch_size}")
print(f"  Number of patches: {patch_embed.n_patches}")

# Check positional embedding
print(f"\nPositional Embedding:")
print(f"  Shape: {model.pos_embedding.shape}")
print(f"  Number of positions: {model.pos_embedding.shape[1]} (1 class token + {patch_embed.n_patches} patches)")

# Check transformer blocks
print(f"\nTransformer Blocks:")
print(f"  Number of blocks: {len(model.transformer)}")
print(f"  Each block contains:")
print(f"    - Layer Normalization")
print(f"    - Multi-Head Self-Attention")
print(f"    - Feed Forward Network")



## 5. Training Example

This section shows how to train the model (requires data directory structure).


In [ ]:
# Training example (commented out - requires data)
# Author: RSK World
# Website: https://rskworld.in

"""
# Load configuration
config = load_config('config.yaml')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay']
)

# Get data loaders (requires data directory)
from utils import get_data_loaders
train_loader, val_loader = get_data_loaders(config)

# Training loop example
for epoch in range(config['training']['num_epochs']):
    model.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
    
    # Validation
    model.eval()
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            # Calculate accuracy...
"""

print("Training code template provided above.")
print("To train, uncomment the code and ensure you have data in the correct directory structure.")



## 6. Inference Example

Example of how to use the trained model for inference.


In [ ]:
# Inference example
# Author: RSK World
# Website: https://rskworld.in

def predict_image(model, image_path, transform, device, top_k=5):
    """
    Predict class for an image
    Author: RSK World
    Website: https://rskworld.in
    """
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Inference
    model.eval()
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        top_probs, top_indices = torch.topk(probabilities, top_k)
    
    return top_probs.cpu().numpy()[0], top_indices.cpu().numpy()[0]

# Example usage (uncomment if you have a trained model and image)
"""
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = get_data_transforms(load_config('config.yaml'), is_train=False)

# Load trained model
checkpoint = torch.load('models/best_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)

# Predict
probs, indices = predict_image(model, 'path/to/image.jpg', transform, device)

print("Top Predictions:")
for i, (prob, idx) in enumerate(zip(probs, indices), 1):
    print(f"{i}. Class {idx}: {prob * 100:.2f}%")
"""

print("Inference code template provided above.")



## Summary

This notebook demonstrated:
1. Vision Transformer model architecture
2. Model forward pass
3. Patch embedding visualization
4. Model components examination
5. Training template
6. Inference template

For more information, visit: https://rskworld.in

**Author:** RSK World  
**Email:** help@rskworld.in  
**Phone:** +91 93305 39277

